In [0]:
dbutils.widgets.remove("year")
dbutils.widgets.text("year", "2025")
year = int(dbutils.widgets.get("year"))
print(f"Running Comtrade ingestion for year: {year}")

In [0]:
import requests
import pandas as pd
from pyspark.sql.types import *
import time

In [0]:
import requests# Load in UN Comtrade country data


ref_url = "https://comtradeapi.un.org/files/v1/app/reference/Reporters.json"

data = requests.get(ref_url).json()

pdf = (
    pd.DataFrame(data.get("results", []))
    .sort_values("reporterCode")
)
pdf = pdf[
    ((pdf['entryExpiredDate'].isnull()) & (pdf['isGroup'] == False))
    | (pdf['reporterCode'] == 490)
]
pdf.loc[pdf['reporterCode'] == 490, 'reporterDesc'] = 'Taiwan (Other Asia, nes)'
pdf.loc[pdf['reporterCode'] == 490, 'reporterCodeIsoAlpha2'] = 'TW'

In [0]:
display(pdf)

In [0]:
# Load in UN Comtrade country data

ref_url = "https://comtradeapi.un.org/files/v1/app/reference/Reporters.json"

data = requests.get(ref_url).json()

pdf = (
    pd.DataFrame(data.get("results", []))
    .sort_values("reporterCode")
)
pdf = pdf[
    ((pdf['entryExpiredDate'].isnull()) & (pdf['isGroup'] == False))
    | (pdf['reporterCode'] == 490)
]
pdf.loc[pdf['reporterCode'] == 490, 'reporterDesc'] = 'Taiwan (Other Asia, nes)'
pdf.loc[pdf['reporterCode'] == 490, 'reporterCodeIsoAlpha2'] = 'TW'
df = spark.createDataFrame(pdf)
df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("bronze_country_un_comtrade")

codes = (
    pdf["reporterCode"]
    .dropna()
    .unique()
    .tolist()
)

print(f"Wrote {len(pdf):,} rows")
print(f"Found {len(codes):,} reporter codes")

# Build once, right after you load pdf from the Reporters.json endpoint
reporter_lookup = dict(zip(pdf["reporterCode"], pdf["reporterDesc"]))

def reporter_name(code):
    return reporter_lookup.get(code, "Unknown")
    

In [0]:
display(pdf)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bronze_comtrade_checkpoint (
  year INT,
  reporter_code INT,
  status STRING,
  updated_at TIMESTAMP
)

In [0]:
groups = split_into_groups(codes, 20)


In [0]:
display(groups)

In [0]:


# Helper functions:
def split_into_groups(codes, n_groups):
    chunk_size = (len(codes) + n_groups - 1) // n_groups  # ceiling division
    return [
        ",".join(map(str, codes[i:i + chunk_size]))
        for i in range(0, len(codes), chunk_size)
    ]

def mark_done(year, code):
    spark.sql(f"""
    INSERT INTO bronze_comtrade_checkpoint
    VALUES ({year}, {code}, 'done', current_timestamp())
    """)

def already_done(year, code):
    return spark.sql(f"""
        SELECT 1
        FROM bronze_comtrade_checkpoint
        WHERE year = {year}
          AND reporter_code = {code}
          AND status = 'done'
        LIMIT 1
    """).count() > 0

comtrade_schema = StructType([
    StructField("typeCode", StringType(), True),
    StructField("freqCode", StringType(), True),
    StructField("refPeriodId", LongType(), True),
    StructField("refYear", LongType(), True),
    StructField("refMonth", LongType(), True),
    StructField("period", StringType(), True),
    StructField("reporterCode", LongType(), True),
    StructField("reporterISO", StringType(), True),
    StructField("reporterDesc", StringType(), True),
    StructField("flowCode", StringType(), True),
    StructField("flowDesc", StringType(), True),
    StructField("partnerCode", LongType(), True),
    StructField("partnerISO", StringType(), True),
    StructField("partnerDesc", StringType(), True),
    StructField("partner2Code", LongType(), True),
    StructField("partner2ISO", StringType(), True),
    StructField("partner2Desc", StringType(), True),
    StructField("cmdCode", StringType(), True),

    StructField("qtyUnitAbbr", StringType(), True),
    StructField("qty", DoubleType(), True),
    StructField("netWgt", DoubleType(), True),

    StructField("cifvalue", DoubleType(), True),
    StructField("fobvalue", DoubleType(), True),
    StructField("primaryValue", DoubleType(), True),
])

URL = "https://comtradeapi.un.org/data/v1/get/C/A/HS"

HEADERS = {
    "Ocp-Apim-Subscription-Key": "2cf570ebb0444d05a046aa6d35ae61e2"
}
def _fetch_group(reporter_code, year, flow, partner_str, max_retries=5):
    params = {
        "reporterCode": reporter_code,
        "partnerCode": partner_str,
        "flowCode": flow,
        "typeCode": "C",
        "period": year,
        "freqCode": "A",
        "cmdCode": "AG4",
        "motCode": 0,
    }
    for attempt in range(max_retries):
        try:
            print(f"Fetching {flow} | reporter {reporter_code} ({reporter_name(reporter_code)}) | partners={len(partner_str.split(','))}")
            response = requests.get(URL, params=params, headers=HEADERS, timeout=120)
            if response.status_code == 429:
                wait = min(60, 2 ** attempt)
                print(f"Rate limited. Waiting {wait}s")
                time.sleep(wait)
                continue
            response.raise_for_status()
            data = response.json()
            df = pd.json_normalize(data.get("data", []))
            print(f"   rows={len(df):,}")
            return df
        except requests.exceptions.RequestException as e:
            wait = min(60, 2 ** attempt)
            print(f"Request failed: {e}. Waiting {wait}s")
            time.sleep(wait)
    print(f"FAILED group after {max_retries} retries: {partner_str[:50]}...")
    return None  # a genuine failure, distinct from a valid-but-empty response


def fetch_comtrade_once(reporter_code, year, flow, partner_codes,
                         initial_groups=2, max_doublings=8,
                         truncation_threshold=100_000):
    """
    Fetches all partner trade data for one reporter/year/flow, batching partners
    into n_groups requests. If any batch returns >= truncation_threshold rows
    (a sign the API silently truncated it), the WHOLE reporter is re-fetched from
    scratch with double the groups (smaller batches), up to max_doublings times.
    """
    n_groups = initial_groups

    for doubling in range(max_doublings):
        groups = split_into_groups(partner_codes, n_groups)
        dfs = []
        truncated = False

        for group_str in groups:
            df_part = _fetch_group(reporter_code, year, flow, group_str)

            if df_part is None:
                print(f"   group failed for reporter {reporter_code} - skipping this group only")
                continue

            if len(df_part) >= truncation_threshold:
                print(f"   reporter {reporter_code}: group hit {len(df_part):,} rows "
                      f"(>= {truncation_threshold:,}) - likely truncated. "
                      f"Restarting with n_groups={n_groups*2}")
                truncated = True
                break  # discard everything collected this attempt, don't trust partial results

            if not df_part.empty:
                dfs.append(df_part)

        if truncated:
            n_groups *= 2
            continue  # retry from scratch with smaller batches

        if not dfs:
            return pd.DataFrame()
        result = pd.concat(dfs, ignore_index=True)
        print(f"TOTAL rows returned for reporter {reporter_code}: {len(result):,} (n_groups={n_groups})")
        return result

    raise RuntimeError(
        f"Reporter {reporter_code}: still hitting the {truncation_threshold:,}-row cap "
        f"after {max_doublings} doublings (final n_groups={n_groups}). "
        f"Investigate this reporter specifically rather than doubling further blindly."
    )


def normalize_comtrade_df(df: pd.DataFrame):

    if df is None:
        return df

    float_cols = [
        "qty", "altQty", "netWgt", "grossWgt",
        "cifvalue", "fobvalue", "primaryValue"
    ]

    for col in float_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    int_cols = [
        "refPeriodId", "refYear", "refMonth",
        "reporterCode", "partnerCode", "partner2Code",
        "motCode", "qtyUnitCode", "altQtyUnitCode",
        "legacyEstimationFlag"
    ]

    for col in int_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

def align_to_schema(pdf, schema_fields):

    for field in schema_fields:
        if field not in pdf.columns:
            pdf[field] = None

    return pdf[schema_fields]

In [0]:
def fetch_comtrade_taiwan(reporter_codes, year, flow,
                           initial_groups=2, max_doublings=7,
                           truncation_threshold=100_000,
                           taiwan_partner_code="490"):
    """
    Fetches Taiwan's trade data via mirror statistics only.

    Taiwan has no reporter code of its own in Comtrade — its trade is
    bundled under partner code 490 ("Other Asia, nes"), and that code's
    REPORTER-side data is restricted to international organizations
    (not available via the public API). So instead of querying Taiwan
    directly, this queries every other country as reporter, with partner
    fixed to 490, and aggregates their declared trade with Taiwan.

    NOTE: this swaps the batching axis vs fetch_comtrade_once — here
    group_str is a batch of REPORTER codes, and the partner is the fixed
    single value 490. This assumes _fetch_group's first argument accepts
    a comma-joined batch string the same way the API already accepts
    batched partner codes (Comtrade's reporterCode field generally
    supports multi-value the same way partnerCode does) — worth a quick
    single-batch smoke test before running the full pull, since this
    helper may not have been exercised in that direction before.

    reporter_codes: full list of reporter codes to query as mirror sources
                     (everyone except 490 itself)
    """
    n_groups = initial_groups
    partner_str = str(taiwan_partner_code)

    for doubling in range(max_doublings):
        groups = split_into_groups(reporter_codes, n_groups)
        dfs = []
        truncated = False

        for group_str in groups:
            # group_str here is a batch of REPORTER codes; partner is fixed to Taiwan (490)
            df_part = _fetch_group(group_str, year, flow, partner_str)

            if df_part is None:
                print(f"   group failed for Taiwan mirror pull - skipping this group only")
                continue

            if len(df_part) >= truncation_threshold:
                print(f"   Taiwan mirror pull: group hit {len(df_part):,} rows "
                      f"(>= {truncation_threshold:,}) - likely truncated. "
                      f"Restarting with n_groups={n_groups*2}")
                truncated = True
                break

            if not df_part.empty:
                dfs.append(df_part)

        if truncated:
            n_groups *= 2
            continue

        if not dfs:
            return pd.DataFrame()
        result = pd.concat(dfs, ignore_index=True)
        print(f"TOTAL rows returned for Taiwan (mirror via partner={partner_str}): "
              f"{len(result):,} (n_groups={n_groups})")
        return result

    raise RuntimeError(
        f"Taiwan mirror pull: still hitting the {truncation_threshold:,}-row cap "
        f"after {max_doublings} doublings (final n_groups={n_groups}). "
        f"Investigate specifically rather than doubling further blindly."
    )

In [0]:
year

In [0]:
fields = [f.name for f in comtrade_schema.fields]
trade = fetch_comtrade_taiwan(codes, 2025, None,
                           initial_groups=8, max_doublings=8,
                           truncation_threshold=100_000,
                           taiwan_partner_code="490")
if not trade.empty:
        trade = normalize_comtrade_df(trade)
        trade = align_to_schema(trade, fields)
        print(f"Preparing to load {len(trade)} rows...")

        spark.createDataFrame(trade, schema=comtrade_schema) \
            .write \
            .mode("append") \
            .saveAsTable("bronze_trade_flows")

In [0]:
year = '2025'
fields = [f.name for f in comtrade_schema.fields]
COUNTER = 0
for code in codes:
    COUNTER+=1
    print(f"\n=== Reporter {code} ===")
    print(f'Iteration {COUNTER} of {len(codes)}')


    if not already_done(year, code):

        try:
            trade = fetch_comtrade_once(code, year, None, partner_codes=codes)

            if not trade.empty:
                trade = normalize_comtrade_df(trade)
                trade = align_to_schema(trade, fields)
                print(f"Preparing to load {len(trade)} rows...")

                spark.createDataFrame(trade, schema=comtrade_schema) \
                    .write \
                    .mode("append") \
                    .saveAsTable("bronze_trade_flows")

            mark_done(year, code)
            print(f"Trade flow done. Loaded {len(trade)} rows")

        except Exception as e:
            print(f"TRADE FLOW FAILED {code}: {e}")
            continue


In [0]:
%sql
SELECT * FROM bronze_trade_flows WHERE rerporter

In [0]:
%sql
SELECT * FROM bronze_trade_flows WHERE refYear = '2025' AND partnerCode = 490

In [0]:
%sql
DELETE FROM bronze_comtrade_checkpoint WHERE year = '2025'